In [6]:
!pip install transformers torch pillow pandas


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd

In [32]:
import os
import torch
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Full image path
        image_path = os.path.join(
            "/content/drive/MyDrive/Colab Notebooks",
            row["image"]
        )

        # Load image
        image = Image.open(image_path).convert("RGB")

        # Process image
        encoding = self.processor(image, return_tensors="pt")
        pixel_values = encoding.pixel_values.squeeze()

        # Process text
        labels = self.processor.tokenizer(
            row["text"],
            padding="max_length",
            max_length=128
        ).input_ids

        labels = torch.tensor(labels)

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

In [12]:
print(UrduOCRDataset)

<class '__main__.UrduOCRDataset'>


In [14]:
from torch.utils.data import DataLoader
from transformers import AdamW

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

optimizer = AdamW(model.parameters(), lr=5e-5)

print("Training batches per epoch:", len(train_loader))
print("Ready to train!")

/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Training batches per epoch: 40
Ready to train!


In [16]:
import os

print(os.path.exists("/content/drive/MyDrive/Colab Notebooks/data/processed/1058.png"))

False


In [33]:
import pandas as pd
import os

# Load labels
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/data/labels.csv")

# Get actual image filenames from the processed folder
folder = "/content/drive/MyDrive/Colab Notebooks/data/processed"
files = sorted(os.listdir(folder), key=lambda x: int(x.split(".")[0]))

# Keep only as many rows as there are images
df = df.iloc[:len(files)].copy()

# Replace image paths with the actual filenames
df["image"] = [f"data/processed/{f}" for f in files]

# Save the corrected CSV
df.to_csv("/content/drive/MyDrive/Colab Notebooks/data/labels.csv", index=False)

print("labels.csv fixed successfully!")
print(df.head())

labels.csv fixed successfully!
                  image                                               text
0  data/processed/0.png  پشاور،بنوں (نمائندہ جنگ،اے ایف پی) بنوں میں اق...
1  data/processed/1.png     اسکے ساتھ ملحقہ علاقے کے عوام نے بجلی و گیس \n
2  data/processed/2.png  نے بجلی وگیس کی لوڈشیڈنگ کیخلاف مظاہرہ کیا۔ مظ...
3  data/processed/3.png  نہیں آ یا۔ آخریہ کونسا مذہب ہے؟کیا اسلام کے نا...
4  data/processed/4.png   ضرور ہے۔ انہوں نے کہا کہ میرا احمدیوں سے کوئی \n


In [34]:
model.eval()

print("=== Model Evaluation on Test Images ===")
print()

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]

        generated_ids = model.generate(pixel_values)

        generated_text = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        actual_text = processor.batch_decode(
            labels,
            skip_special_tokens=True
        )

        for pred, actual in zip(generated_text, actual_text):
            total += 1

            if pred.strip() == actual.strip():
                correct += 1

            print("Predicted:", pred)
            print("Actual   :", actual)
            print()

accuracy = (correct / total) * 100 if total > 0 else 0

print(f"Accuracy: {accuracy:.1f}% ({correct}/{total} correct)")


=== Model Evaluation on Test Images ===



/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1168: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Predicted: 34-UL,KBG BI 6.9
Actual   : ملک کی سیاسی صورتحال،بالخصوص کراچی میں امن واماں کی صورتحال، 


Predicted: 192-6656.206092-62-3 V
Actual   : بارے میں حکومتی لائحہ عمل سے 3 روز میں عدالت 


Predicted: SR/8 4:30/03152 PLUSED
Actual   : نے اپنے ایک بیان میں کہا ہے کہ انہیں اس 


Predicted: GSTALONBUSEU$2.02
Actual   : 75 روپیہ اورتھری فیز کنکشن پر 150روپیہ ماہانہ لازمی چارجز 


Predicted: SR4
Actual   : باغ فرنیچر مارکیٹ میں مارکیٹوں کے نمائندوں کا ایک ہنگامی 


Predicted: 206-J.EL.ST.VE15 ZK
Actual   : روز گزر گئے تفتیشی افسر کو نہیں بھیجا گیا، ہم 


Predicted: 0.22 JALAZTULA
Actual   : آئے ہیں وزیر داخلہ کی وکالت کیلئے نہیں آئے۔ سابق 


Predicted: -SUT2-6U FEJSLUUSUT
Actual   : فی یونٹ جبکہ 6سے 500کے وی تک بجلی کے ا 


Predicted: ***
Actual   : کی۔ وزیراعلیٰ سندھ نے بتایا کہ صوبہ سندھ حالیہ بارشوں 


Predicted: 2,5V6 ZCLUZE6%JLOTRATE JJARA
Actual   : یونٹ ہوگی صنعتی شعبہ کیلئے 6کلو واٹ تک 8روپیہ 96پیسے 


Predicted: SUPPLIPTOTAL GST @6%
Actual   : میں پانچ سیکرٹریوں کو تفتیش کے ل

In [35]:
from google.colab import drive

drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/SI26-urdu-ocr-model"

model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print("Model saved successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model saved successfully!


In [24]:
from transformers import TrOCRProcessor
import torch

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")

dataset = UrduOCRDataset(
    "/content/drive/MyDrive/Colab Notebooks/data/labels.csv",
    processor
)

sample = dataset[0]

print("Sample pixel_values shape:", sample["pixel_values"].shape)
print("Sample labels shape:", sample["labels"].shape)
print("Dataset is working correctly!")

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, test_size]
)

print("Training samples:", train_size)
print("Testing samples:", test_size)

Dataset loaded: 200 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!
Training samples: 160
Testing samples: 40


In [25]:
from torch.utils.data import DataLoader
from transformers import AdamW

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

optimizer = AdamW(model.parameters(), lr=5e-5)

print("Ready to train!")

Ready to train!


/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [31]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-"*30)

    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} complete | Average Loss: {avg_loss:.4f}")

print("Training complete!")


Epoch 1/3
------------------------------
Batch 0/40 | Loss: 17.2685
Batch 10/40 | Loss: 17.5014
Batch 20/40 | Loss: 17.7274
Batch 30/40 | Loss: 16.6815
Epoch 1 complete | Average Loss: 17.4106

Epoch 2/3
------------------------------
Batch 0/40 | Loss: 17.3185
Batch 10/40 | Loss: 17.3317
Batch 20/40 | Loss: 17.5530
Batch 30/40 | Loss: 17.7463
Epoch 2 complete | Average Loss: 17.3922

Epoch 3/3
------------------------------
Batch 0/40 | Loss: 17.4300
Batch 10/40 | Loss: 17.5213
Batch 20/40 | Loss: 17.7252
Batch 30/40 | Loss: 17.1946
Epoch 3 complete | Average Loss: 17.4058
Training complete!


In [30]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

# Load processor
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")

# Load model
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed")

# Move model to GPU
model = model.to(device)

# Configure model
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print("Model loaded successfully!")

Using device: cuda


Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully!


In [27]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/data/labels.csv")
print(df.head())

                  image                                               text
0  data/processed/0.png  پشاور،بنوں (نمائندہ جنگ،اے ایف پی) بنوں میں اق...
1  data/processed/1.png     اسکے ساتھ ملحقہ علاقے کے عوام نے بجلی و گیس \n
2  data/processed/2.png  نے بجلی وگیس کی لوڈشیڈنگ کیخلاف مظاہرہ کیا۔ مظ...
3  data/processed/3.png  نہیں آ یا۔ آخریہ کونسا مذہب ہے؟کیا اسلام کے نا...
4  data/processed/4.png   ضرور ہے۔ انہوں نے کہا کہ میرا احمدیوں سے کوئی \n


In [17]:
import os

folder = "/content/drive/MyDrive/Colab Notebooks/data/processed"

files = sorted(os.listdir(folder))

print("Total images:", len(files))
print("First 20 images:")
print(files[:20])

print("\nLast 20 images:")
print(files[-20:])

Total images: 200
First 20 images:
['0.png', '1.png', '10.png', '100.png', '101.png', '102.png', '103.png', '104.png', '105.png', '106.png', '107.png', '108.png', '109.png', '11.png', '110.png', '111.png', '112.png', '113.png', '114.png', '115.png']

Last 20 images:
['81.png', '82.png', '83.png', '84.png', '85.png', '86.png', '87.png', '88.png', '89.png', '9.png', '90.png', '91.png', '92.png', '93.png', '94.png', '95.png', '96.png', '97.png', '98.png', '99.png']


In [18]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/data/labels.csv")

print(df.head(10))

                      image                                               text
0      data/processed/0.png  پشاور،بنوں (نمائندہ جنگ،اے ایف پی) بنوں میں اق...
1      data/processed/1.png     اسکے ساتھ ملحقہ علاقے کے عوام نے بجلی و گیس \n
2     data/processed/10.png  نے بجلی وگیس کی لوڈشیڈنگ کیخلاف مظاہرہ کیا۔ مظ...
3    data/processed/100.png  نہیں آ یا۔ آخریہ کونسا مذہب ہے؟کیا اسلام کے نا...
4   data/processed/1000.png   ضرور ہے۔ انہوں نے کہا کہ میرا احمدیوں سے کوئی \n
5  data/processed/10000.png  ڈالر کی قدر میں اضافہ ہوگیا تاہم یورپین ممالک ...
6  data/processed/10001.png  اہم کرنسیوں بشمول یورو اور برطانوی پاؤنڈ کی قد...
7  data/processed/10002.png  کمی ریکارڈ کی گئی۔ منگل کو امریکی ڈالر کی قیمت \n
8  data/processed/10003.png  فروخت اور قیمت خرید میں بالترتیب 21 پیسے اور 2...
9  data/processed/10004.png   پیسے کی تیزی رہی، تاہم یورو کی قیمت فروخت میں \n


In [20]:
import pandas as pd
import os

# Load labels
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/data/labels.csv")

# Get actual image filenames from the processed folder
folder = "/content/drive/MyDrive/Colab Notebooks/data/processed"
files = sorted(os.listdir(folder), key=lambda x: int(x.split(".")[0]))

# Keep only as many rows as there are images
df = df.iloc[:len(files)].copy()

# Replace image paths with the actual filenames
df["image"] = [f"data/processed/{f}" for f in files]

# Save the corrected CSV
df.to_csv("/content/drive/MyDrive/Colab Notebooks/data/labels.csv", index=False)

print("labels.csv fixed successfully!")
print(df.head())

labels.csv fixed successfully!
                  image                                               text
0  data/processed/0.png  پشاور،بنوں (نمائندہ جنگ،اے ایف پی) بنوں میں اق...
1  data/processed/1.png     اسکے ساتھ ملحقہ علاقے کے عوام نے بجلی و گیس \n
2  data/processed/2.png  نے بجلی وگیس کی لوڈشیڈنگ کیخلاف مظاہرہ کیا۔ مظ...
3  data/processed/3.png  نہیں آ یا۔ آخریہ کونسا مذہب ہے؟کیا اسلام کے نا...
4  data/processed/4.png   ضرور ہے۔ انہوں نے کہا کہ میرا احمدیوں سے کوئی \n


In [19]:
print(df.tail(10))


                       image  \
190  data/processed/1111.png   
191  data/processed/1112.png   
192  data/processed/1113.png   
193  data/processed/1114.png   
194  data/processed/1115.png   
195  data/processed/1116.png   
196  data/processed/1117.png   
197  data/processed/1118.png   
198  data/processed/1119.png   
199   data/processed/112.png   

                                                  text  
190  کی عدم دلچسپی، تجارتی شہرکے کاروباری مستقبل او...  
191         جان و مال کے تحفظ پر غورکیا جائے گا۔ اس \n  
192  سلسلے میں آل کراچی تاجر اتحاد کے چیئرمین عتیق ...  
193     نے پریس کو جاری کیے گئے ایک اعلامیے میں کہا \n  
194  کہ شہری زندگی کی مہذب شناخت ختم ہوگئی،کراچی جگ...  
195   جنگل کی تصویر پیش کر رہا ہے۔ ٹارگٹ کلنگ، بھتہ \n  
196  مافیا، ڈاکووٴں اور لٹیروں نے کراچی کی تجارتی ح...  
197    تباہ و برباد کرکے رکھ دیا ہے۔ کراچی مکمل طور \n  
198  پر ڈاکووٴں، لٹیروں اور بھتہ مافیا کی ریاست بن ...  
199         بلکہ معاف کرنے والے ہیں، آپ ﷺ نے تو فتح \n  


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import transformers
print(transformers.__version__)

4.41.2


In [2]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")

model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
)

print("Processor and model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Processor and model loaded successfully!
